# Batch extraction of Imaris IS surfaces

Batch extraction of the time-offset corrected Imaris Surface csv file.
Imaris surface entries for IS-250, IS-500 and IS-auto.

This skript:
- Loops over all CSV files in INPUT_DIR.
- For every input CSV, tries to extract each target component:
    IS-250, IS-500, IS-auto
- Writes one output CSV per successfully processed component:
    originalFilename_IS-250.csv
    originalFilename_IS-500.csv
    originalFilename_IS-auto.csv
- Normalizes summed volume to the t=0 / first-time-point summed volume, not to the maximum volume.
- Skips a file/component if fewer than MIN_ENTRIES rows are found for that component.
- Writes an exclusion log indicating which file/component was skipped and why.

Adapted from: Prelim_IS-volume-time_260223.ipynb
"""

Set `INPUT_DIR` in the first code cell, then run the cell. Outputs are written to `batch_IS_outputs`.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd


# =========================
# USER SETTINGS
# =========================

INPUT_DIR = Path(r" *** \SampleData\Individual-run_TestData_Script1-3\run1_ImarisSurfaces\Time_offset_corrected")

# Recommended: keep outputs in a separate folder so they are not reprocessed in later batch runs.
OUTPUT_DIR = INPUT_DIR / "batch_IS_outputs"

TARGET_COMPONENTS = ["IS-250", "IS-500", "IS-auto"]

# Minimum number of raw surface entries required for a given file/component.
MIN_ENTRIES = 4

# File search pattern. Usually "*.csv".
CSV_GLOB = "*.csv"

# If True, existing output CSVs/logs with the same name will be overwritten.
OVERWRITE = True

# Volume normalization settings.
# The script first tries to normalize to the summed volume at Time == 0.
# If no exact t=0 row exists, it can fall back to the first available time point
# after sorting by the main time column. This is useful if Imaris exports start at
# frame/time 1 instead of 0.
NORMALIZE_TO_TIME_ZERO = True
BASELINE_TIME_VALUE = 0
BASELINE_TIME_TOLERANCE = 1e-9
FALLBACK_TO_FIRST_TIMEPOINT_IF_NO_T0 = True


# =========================
# HELPER FUNCTIONS
# =========================

class SkipComponent(Exception):
    """Used for expected skip cases, e.g. fewer than MIN_ENTRIES entries."""
    def __init__(self, reason, entry_count=None):
        super().__init__(reason)
        self.reason = reason
        self.entry_count = entry_count


def find_imaris_header_row(path):
    """
    Imaris CSV exports often contain a few metadata lines before the real header.
    This scans the file and returns the line index of the true header row.
    """
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            line_l = line.lower()
            if ("component name" in line_l) and ("time" in line_l) and ("volume" in line_l):
                return i
    raise ValueError("Could not find the Imaris table header row in the CSV file.")


def find_column(columns, candidates=None, contains_all=None, prefer_exact=None):
    """
    Flexible column finder:
    - prefer_exact: list of exact names to try first
    - candidates: list of exact names to try next
    - contains_all: list of substrings that must all be present, case-insensitive
    """
    cols = list(columns)
    cols_norm = {c: c.strip().lower() for c in cols}

    if prefer_exact:
        for name in prefer_exact:
            for c in cols:
                if c.strip().lower() == name.strip().lower():
                    return c

    if candidates:
        for name in candidates:
            for c in cols:
                if c.strip().lower() == name.strip().lower():
                    return c

    if contains_all:
        for c, c_norm in cols_norm.items():
            if all(x.lower() in c_norm for x in contains_all):
                return c

    return None


def is_generated_output_file(path, target_components):
    """
    Avoids re-processing files that were generated by this batch script.
    Example excluded stems:
        sample_IS-250
        sample_IS-500
        sample_IS-auto
    """
    stem_lower = path.stem.lower()
    output_suffixes = tuple(f"_{component.lower()}" for component in target_components)
    return stem_lower.endswith(output_suffixes)


def make_output_path(input_path, target_component):
    """
    Creates output path:
        originalFilename_IS-250.csv
    inside OUTPUT_DIR.
    """
    return OUTPUT_DIR / f"{input_path.stem}_{target_component}.csv"


def read_imaris_csv(csv_path):
    """Read one Imaris CSV and remove empty trailing columns."""
    header_row = find_imaris_header_row(csv_path)
    df = pd.read_csv(csv_path, skiprows=header_row)

    # Remove empty trailing columns like "Unnamed: ..."
    df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]

    return df


def extract_component_table(csv_path, target_component, min_entries=5):
    """
    Extract one component from one Imaris CSV and return the aggregated table.

    The output table contains:
    - all relevant time columns if present, e.g. Time plus Time [min] or Time [s]
    - summed volume per time point
    - normalized volume
    - averaged intensity means per channel
    - propagated SD per channel
    - SEM between object means per channel
    - number of objects per channel/time point
    """
    df = read_imaris_csv(csv_path)

    # Find relevant columns robustly, because Imaris may include units in names.
    component_col = find_column(
        df.columns,
        prefer_exact=["Component Name", "component name"]
    )
    # Imaris can export several useful time columns:
    # - "Time" or "Time Index": usually the frame/time-point number
    # - "Time [min]" / "Time(min)": physical elapsed time in minutes
    # - "Time [s]" / "Time(s)": physical elapsed time in seconds
    #
    # Keep "Time" as the main grouping column, but also carry any physical
    # time-with-unit column into the output if it exists.
    time_col = find_column(
        df.columns,
        prefer_exact=["Time", "time"],
        candidates=["Time Index"]
    )

    time_unit_candidates = [
        "Time [min]", "Time(min)", "Time (min)", "Time min",
        "time [min]", "time(min)", "time (min)", "time min",
        "Time [s]", "Time(s)", "Time (s)", "Time s",
        "time [s]", "time(s)", "time (s)", "time s",
        "Time [sec]", "Time(sec)", "Time (sec)", "Time sec",
        "Time [seconds]", "Time(seconds)", "Time (seconds)", "Time seconds",
    ]

    # Preserve the original column order and keep all matching unit columns.
    time_unit_cols = []
    for col in df.columns:
        col_clean = str(col).strip()
        col_norm = col_clean.lower().replace(" ", "")
        is_known_unit_time = any(
            col_clean.lower() == candidate.lower()
            for candidate in time_unit_candidates
        )
        is_time_min_or_seconds = (
            col_norm.startswith("time[")
            and col_norm.endswith("]")
            and any(unit in col_norm for unit in ["min", "s", "sec", "second"])
        )
        if (is_known_unit_time or is_time_min_or_seconds) and col != time_col:
            if col not in time_unit_cols:
                time_unit_cols.append(col)

    volume_col = find_column(
        df.columns,
        prefer_exact=["Volume", "volume"],
        contains_all=["volume"]
    )

    if component_col is None:
        raise ValueError(f"Could not find a 'Component Name' column. Columns: {list(df.columns)}")
    if time_col is None:
        # Fallback: if there is no frame-like "Time" column, group directly by the
        # first physical time column found, e.g. Time [min] or Time [s].
        if time_unit_cols:
            time_col = time_unit_cols.pop(0)
        else:
            raise ValueError(f"Could not find a time column. Columns: {list(df.columns)}")
    if volume_col is None:
        raise ValueError(f"Could not find a volume column. Columns: {list(df.columns)}")

    # Filter to the selected component.
    mask_component = (
        df[component_col]
        .astype(str)
        .str.strip()
        .str.upper()
        == target_component.upper()
    )
    df_component = df.loc[mask_component].copy()

    raw_entry_count = len(df_component)

    if raw_entry_count == 0:
        available_components = sorted(df[component_col].dropna().astype(str).str.strip().unique())
        raise SkipComponent(
            reason=(
                f"No rows found for Component Name == '{target_component}'. "
                f"Available components: {available_components}"
            ),
            entry_count=0
        )

    if raw_entry_count < min_entries:
        raise SkipComponent(
            reason=f"Fewer than {min_entries} entries for Component Name == '{target_component}'.",
            entry_count=raw_entry_count
        )

    # Convert essential columns to numeric and remove invalid rows.
    df_component[time_col] = pd.to_numeric(df_component[time_col], errors="coerce")
    for time_unit_col in time_unit_cols:
        df_component[time_unit_col] = pd.to_numeric(df_component[time_unit_col], errors="coerce")
    df_component[volume_col] = pd.to_numeric(df_component[volume_col], errors="coerce")
    df_component = df_component.dropna(subset=[time_col, volume_col])

    valid_entry_count = len(df_component)
    if valid_entry_count < min_entries:
        raise SkipComponent(
            reason=(
                f"Fewer than {min_entries} valid numeric entries after cleaning "
                f"for Component Name == '{target_component}'."
            ),
            entry_count=valid_entry_count
        )

    # Base output: total volume per time point.
    total_volume_per_time = (
        df_component.groupby(time_col, as_index=False)[volume_col]
        .sum()
        .sort_values(by=time_col)
    )

    # Add physical time-with-unit columns, if present.
    # Use first() rather than grouping by these float columns directly, to avoid
    # accidentally splitting one frame into multiple rows because of tiny
    # floating-point export differences.
    for time_unit_col in time_unit_cols:
        time_unit_per_time = (
            df_component.groupby(time_col, as_index=False)[time_unit_col]
            .first()
        )
        total_volume_per_time = total_volume_per_time.merge(
            time_unit_per_time,
            on=time_col,
            how="left"
        )

    # Keep all time columns at the front.
    front_cols = [time_col] + [c for c in time_unit_cols if c in total_volume_per_time.columns]
    other_cols = [c for c in total_volume_per_time.columns if c not in front_cols]
    total_volume_per_time = total_volume_per_time[front_cols + other_cols]

    # Add normalized volume column.
    # Old behavior was max-normalization:
    #     Normalized Volume = summed volume at time t / maximum summed volume in this file/component
    # New behavior is baseline normalization to t=0 / first time point:
    #     Normalized Volume = summed volume at time t / summed volume at t=0
    #
    # This preserves cases where mock or untreated samples increase above their starting volume:
    # values can become >1 instead of being forced to have a maximum of exactly 1.
    time_values_numeric = pd.to_numeric(total_volume_per_time[time_col], errors="coerce")
    volume_values_numeric = pd.to_numeric(total_volume_per_time[volume_col], errors="coerce")

    if NORMALIZE_TO_TIME_ZERO:
        t0_mask = np.isclose(
            time_values_numeric,
            BASELINE_TIME_VALUE,
            atol=BASELINE_TIME_TOLERANCE,
            equal_nan=False
        )

        if t0_mask.any():
            baseline_vol = volume_values_numeric.loc[t0_mask].iloc[0]
        elif FALLBACK_TO_FIRST_TIMEPOINT_IF_NO_T0 and len(volume_values_numeric) > 0:
            # total_volume_per_time is already sorted by time_col above.
            baseline_vol = volume_values_numeric.iloc[0]
        else:
            baseline_vol = pd.NA

        if pd.isna(baseline_vol) or baseline_vol == 0:
            total_volume_per_time["Normalized Volume"] = pd.NA
        else:
            total_volume_per_time["Normalized Volume"] = volume_values_numeric / baseline_vol

    else:
        # Optional fallback to the previous behavior, if NORMALIZE_TO_TIME_ZERO is set to False.
        max_vol = volume_values_numeric.max()
        if pd.isna(max_vol) or max_vol == 0:
            total_volume_per_time["Normalized Volume"] = pd.NA
        else:
            total_volume_per_time["Normalized Volume"] = volume_values_numeric / max_vol

    # Find all Intensity Mean / Intensity StdDev columns by channel.
    # Example columns:
    # "Intensity Mean Ch=1 Img=1"
    # "Intensity StdDev Ch=1 Img=1"
    mean_pattern = re.compile(r"^Intensity Mean Ch=(\d+)\b.*", re.IGNORECASE)
    std_pattern = re.compile(r"^Intensity StdDev Ch=(\d+)\b.*", re.IGNORECASE)

    mean_cols_by_ch = {}
    std_cols_by_ch = {}

    for col in df_component.columns:
        col_clean = str(col).strip()
        m_mean = mean_pattern.match(col_clean)
        m_std = std_pattern.match(col_clean)

        if m_mean:
            ch = int(m_mean.group(1))
            mean_cols_by_ch[ch] = col

        if m_std:
            ch = int(m_std.group(1))
            std_cols_by_ch[ch] = col

    channels_found = sorted(set(mean_cols_by_ch.keys()) & set(std_cols_by_ch.keys()))

    # Aggregate mean intensity + propagated error for each channel.
    for ch in channels_found:
        mean_col_ch = mean_cols_by_ch[ch]
        std_col_ch = std_cols_by_ch[ch]

        tmp = df_component[[time_col, mean_col_ch, std_col_ch]].copy()
        tmp[mean_col_ch] = pd.to_numeric(tmp[mean_col_ch], errors="coerce")
        tmp[std_col_ch] = pd.to_numeric(tmp[std_col_ch], errors="coerce")
        tmp = tmp.dropna(subset=[time_col, mean_col_ch, std_col_ch])

        if tmp.empty:
            continue

        grp = tmp.groupby(time_col)

        # Average of object means at each time point.
        mean_of_means = grp[mean_col_ch].mean()

        # Number of objects used at each time point.
        n_objects = grp[mean_col_ch].count()

        # Propagated SD of the averaged mean:
        # sigma_mean = sqrt(sum_i sigma_i^2) / n
        propagated_sd = grp[std_col_ch].apply(
            lambda s: np.sqrt(np.sum(np.square(s))) / len(s)
        )

        # Additional diagnostic:
        # SEM of the spread between object means.
        between_obj_sd = grp[mean_col_ch].std(ddof=1)
        sem_between_obj_means = between_obj_sd / np.sqrt(n_objects)

        ch_out = pd.DataFrame({
            time_col: mean_of_means.index,
            f"Intensity Mean Ch={ch} (avg)": mean_of_means.values,
            f"Intensity Mean Ch={ch} (propagated SD)": propagated_sd.values,
            f"Intensity Mean Ch={ch} (propagated SEM)": sem_between_obj_means.values,
            f"Intensity Mean Ch={ch} (n objects)": n_objects.values,
        })

        total_volume_per_time = total_volume_per_time.merge(ch_out, on=time_col, how="left")

    total_volume_per_time = (
        total_volume_per_time
        .sort_values(by=time_col)
        .reset_index(drop=True)
    )

    return total_volume_per_time, raw_entry_count, valid_entry_count


def batch_process_folder():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    processed_records = []
    excluded_records = []
    error_records = []

    input_files = sorted(INPUT_DIR.glob(CSV_GLOB))

    # Avoid processing outputs from previous runs if OUTPUT_DIR is inside INPUT_DIR.
    input_files = [
        p for p in input_files
        if p.is_file() and not is_generated_output_file(p, TARGET_COMPONENTS)
    ]

    if not input_files:
        print(f"No input files found in:\n{INPUT_DIR}")
        return

    for csv_path in input_files:
        for target_component in TARGET_COMPONENTS:
            output_path = make_output_path(csv_path, target_component)

            if output_path.exists() and not OVERWRITE:
                excluded_records.append({
                    "file": csv_path.name,
                    "target_component": target_component,
                    "entry_count": "",
                    "reason": f"Output already exists and OVERWRITE is False: {output_path.name}",
                })
                continue

            try:
                result_table, raw_count, valid_count = extract_component_table(
                    csv_path=csv_path,
                    target_component=target_component,
                    min_entries=MIN_ENTRIES
                )

                result_table.to_csv(output_path, index=False)

                processed_records.append({
                    "file": csv_path.name,
                    "target_component": target_component,
                    "raw_entry_count": raw_count,
                    "valid_entry_count": valid_count,
                    "output_file": output_path.name,
                })

            except SkipComponent as exc:
                excluded_records.append({
                    "file": csv_path.name,
                    "target_component": target_component,
                    "entry_count": exc.entry_count,
                    "reason": exc.reason,
                })

            except Exception as exc:
                error_records.append({
                    "file": csv_path.name,
                    "target_component": target_component,
                    "reason": repr(exc),
                })

    # Save detailed machine-readable processing log.
    log_rows = []

    for r in processed_records:
        log_rows.append({
            "status": "processed",
            "file": r["file"],
            "target_component": r["target_component"],
            "raw_entry_count": r["raw_entry_count"],
            "valid_entry_count": r["valid_entry_count"],
            "output_file": r["output_file"],
            "reason": "",
        })

    for r in excluded_records:
        log_rows.append({
            "status": "excluded",
            "file": r["file"],
            "target_component": r["target_component"],
            "raw_entry_count": r.get("entry_count", ""),
            "valid_entry_count": "",
            "output_file": "",
            "reason": r["reason"],
        })

    for r in error_records:
        log_rows.append({
            "status": "error",
            "file": r["file"],
            "target_component": r["target_component"],
            "raw_entry_count": "",
            "valid_entry_count": "",
            "output_file": "",
            "reason": r["reason"],
        })

    log_df = pd.DataFrame(log_rows)
    log_csv_path = OUTPUT_DIR / "processing_log.csv"
    log_df.to_csv(log_csv_path, index=False)

    # Save human-readable exclusion/error text file.
    excluded_txt_path = OUTPUT_DIR / "excluded_files.txt"
    with open(excluded_txt_path, "w", encoding="utf-8") as f:
        f.write("Excluded or failed file/component combinations\n")
        f.write("=============================================\n\n")

        if not excluded_records and not error_records:
            f.write("None.\n")
        else:
            if excluded_records:
                f.write("Excluded because of missing component, too few entries, or existing output:\n")
                for r in excluded_records:
                    f.write(
                        f"- {r['file']} | {r['target_component']} | "
                        f"entries: {r.get('entry_count', '')} | {r['reason']}\n"
                    )
                f.write("\n")

            if error_records:
                f.write("Errors:\n")
                for r in error_records:
                    f.write(f"- {r['file']} | {r['target_component']} | {r['reason']}\n")

    # Console summary.
    print("\nBatch processing finished.")
    print(f"Input folder:  {INPUT_DIR}")
    print(f"Output folder: {OUTPUT_DIR}")
    print(f"Processed file/component combinations: {len(processed_records)}")
    print(f"Excluded file/component combinations:  {len(excluded_records)}")
    print(f"Errors:                                {len(error_records)}")
    print(f"\nDetailed log: {log_csv_path}")
    print(f"Excluded files text log: {excluded_txt_path}")

    if excluded_records:
        print("\nExcluded:")
        for r in excluded_records:
            print(
                f"- {r['file']} | {r['target_component']} | "
                f"entries: {r.get('entry_count', '')} | {r['reason']}"
            )

    if error_records:
        print("\nErrors:")
        for r in error_records:
            print(f"- {r['file']} | {r['target_component']} | {r['reason']}")


if __name__ == "__main__":
    batch_process_folder()
